### Importing Libraries

In [1]:
import sys
import time
import warnings
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from mlflow.data import from_pandas
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

from churnlabs.core.config import PROJECT_ROOT

### Warnings Configuration

In [2]:
# Suppress MLflow Warnings Only
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")

### MLflow Configuration

In [3]:
# This sets the tracking URI for MLflow
# Instead of using the default local ./mlruns folder,
# It configure MLflow to use a SQLite database stored inside the project directory
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")

### Custom Seaborn Plot Style

In [4]:
# This snippet allow us to create a custom style of plots in Seaborn
sns.set_style('ticks')
sns.set_theme('paper')

### Importing Parquet Data

In [5]:
from churnlabs.data.loaders import load_processed_data
churn_data = load_processed_data()
churn_data.head()

,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.850000,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950001,1889.500000,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.849998,108.150002,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.299999,1840.750000,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.699997,151.649994,Yes


### Copy of Original DataFrame

In [6]:
# Making a copy of Original DataFrame
df = churn_data.copy()

### Splitting the Data

In [7]:
# Splitting Data into Training and Testing Set
from churnlabs.features.split import split_data
X_train, X_test, y_train, y_test = split_data(df)

### Encoding Target Variable

In [8]:
# Encoding Target Variable (Yes/No -> 1/0)
from churnlabs.models.encoder import target_encoder
y_train, y_test = target_encoder(y_train, y_test)

### Creating a Pipeline

In [9]:
# Importing Libraries
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_validate, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report, ConfusionMatrixDisplay, precision_recall_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
# Importing Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [11]:
# Transformation for Categorical Columns
cat_cols = X_train.select_dtypes(include='category').columns

cat_trf = Pipeline(steps=[
    ('ohe', OneHotEncoder(sparse_output=False, drop='first'))
])

In [12]:
# Transformation for Numerical Columns
num_cols = [col for col in X_train.select_dtypes(include='number').columns if col != 'seniorcitizen']

num_trf = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [13]:
# Column Transformation
ctf = ColumnTransformer(transformers=[
    ('categorical', cat_trf, cat_cols),
    ('numerical', num_trf, num_cols)
], remainder='passthrough', n_jobs=-1)

In [14]:
# Importing DummyClassifier Model
dummy = DummyClassifier(strategy='most_frequent', random_state=42)

In [15]:
# Pipeline
pipe = Pipeline(steps=[
    ('preprocessor', ctf),
    ('model', dummy)
])

In [16]:
# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
# Cross-Validation
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}

cv = cross_validate(estimator=pipe, X=X_train, y=y_train, cv=skf, scoring=scoring, n_jobs=-1)

In [18]:
# Cross Validation Result
results = {metric.replace('test_', ''): [np.mean(scores), np.std(scores)] for metric, scores in cv.items() if metric.startswith('test')}
results_df = pd.DataFrame(results, index=['mean', 'std']).T
results_df

,mean,std
accuracy,0.734222,0.0
precision,0.000000,0.0
recall,0.000000,0.0
f1,0.000000,0.0
roc_auc,0.500000,0.0
pr_auc,0.265778,0.0


**What `DummyClassifier` helps us check?**
- The `DummyClassifier` gives us a basic baseline to compare against real models.
- It predicts the majority class (`non-churn`) for every customer.
```
# Class Distribution
churn
No     0.7342
Yes    0.2657
```
```
# Classification Metrics
accuracy    73%
precision   0
recall      0
f1          0
roc-auc     0.5
pr-auc      0.26
```
- Since the dataset contains approximately 73% non-churn customers, the model achieves 73% accuracy.
- However, it fails to identify any churners, resulting in zero precision, recall, and F1 score.
- The ROC AUC of 0.5 confirms that the model has no predictive power and performs equivalent to random guessing.
- The `DummyClassifier` makes exactly the same prediction in every fold.
- So every fold produces identical metrics, and that's why standard deviation is equal to 0.
- This confirms our pipeline and cross-validation setup behave as expected and no obvious leakage exists.
- Now any real model must surpass this benchmark to be considered a good performer.
```
# Real Model Expectations
Achieve PR AUC > 0.26
Achieve ROC AUC > 0.5
Achieve Precision > 0
Achieve Recall > 0
Achieve F1 Score > 0
```
- If a trained model cannot outperform this baseline, it is not useful.

---